# NB17：企業案例 — 智能數據分析助理（Text-to-SQL + BI Agent）

## 學習目標

本筆記本以一家台灣連鎖零售業為情境，示範如何建構一個讓業務分析師和門店管理者能以**自然語言**查詢資料、產生洞察、生成報表的 AI 助理，無需撰寫 SQL。

| 主題 | 說明 |
|------|------|
| Text-to-SQL | 將自然語言轉換為安全的 SQL 查詢 |
| Schema 理解 | 自動抽取資料庫結構並注入至 LLM 提示 |
| SQL 驗證與錯誤復原 | 攔截危險操作，自動修正錯誤 SQL |
| 結果解讀 | 將查詢結果轉為自然語言洞察 |
| 多步驟推理 | 鏈式查詢與複雜分析問題拆解 |
| 安全邊界 | 防止 SQL 注入與權限控制 |

---

## 情境設定

**「全聯智慧零售」** 擁有全台 50 家門店、3 年交易資料。公司希望：
- 業務分析師：用白話文查詢銷售趨勢，不用等 IT 部門寫報表
- 門店店長：隨時查看自家門店績效，與全台平均比較
- 行銷團隊：快速找出暢銷品類與潛力商品

```
使用者問題
    ↓
Schema 注入 + Prompt 構建
    ↓
LLM (GPT-4o-mini) → SQL 生成
    ↓
SQL 驗證器（攔截危險操作）
    ↓
SQLite 執行器 → DataFrame
    ↓
結果解讀器 → 自然語言洞察
    ↓
使用者看到業務洞察
```

---
## Part 0：環境設定與資料庫建立

建立一個包含 **50 家門店、200 種商品、5000+ 筆交易** 的記憶體 SQLite 資料庫，模擬真實零售業資料。

In [ ]:
import os
import sqlite3
import random
import json
import re
import time
from datetime import date, timedelta, datetime
from typing import Optional, Tuple
from dataclasses import dataclass, field

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# 中文字型設定（macOS / Linux 皆可用）
matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

CHAT_MODEL = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'
random.seed(42)

print('✓ 環境初始化完成')
print(f'  Chat model  : {CHAT_MODEL}')
print(f'  Embed model : {EMBED_MODEL}')

In [ ]:
# ─── 建立記憶體資料庫 ───────────────────────────────────────────────────────
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# ── DDL ──────────────────────────────────────────────────────────────────────
cursor.executescript("""
CREATE TABLE stores (
    store_id    INTEGER PRIMARY KEY,
    store_name  TEXT NOT NULL,
    city        TEXT NOT NULL,
    district    TEXT NOT NULL,
    area_sqm    INTEGER NOT NULL,
    opened_date TEXT NOT NULL
);

CREATE TABLE products (
    product_id    INTEGER PRIMARY KEY,
    name          TEXT NOT NULL,
    category      TEXT NOT NULL,
    cost_price    REAL NOT NULL,
    selling_price REAL NOT NULL
);

CREATE TABLE transactions (
    txn_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    store_id       INTEGER NOT NULL,
    product_id     INTEGER NOT NULL,
    quantity       INTEGER NOT NULL,
    amount         REAL NOT NULL,
    txn_date       TEXT NOT NULL,
    payment_method TEXT NOT NULL,
    FOREIGN KEY (store_id)   REFERENCES stores(store_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE employees (
    emp_id    INTEGER PRIMARY KEY,
    store_id  INTEGER NOT NULL,
    name      TEXT NOT NULL,
    role      TEXT NOT NULL,
    hire_date TEXT NOT NULL,
    salary    INTEGER NOT NULL,
    FOREIGN KEY (store_id) REFERENCES stores(store_id)
);
""")
print('✓ 資料表建立完成')

In [ ]:
# ─── 產生真實感模擬資料 ────────────────────────────────────────────────────

# 門店資料
cities_districts = [
    ('台北市','信義區'),('台北市','大安區'),('台北市','中山區'),('台北市','松山區'),
    ('台北市','內湖區'),('台北市','士林區'),('台北市','北投區'),('台北市','文山區'),
    ('新北市','板橋區'),('新北市','新莊區'),('新北市','中和區'),('新北市','永和區'),
    ('新北市','土城區'),('新北市','三重區'),('新北市','淡水區'),('新北市','新店區'),
    ('桃園市','桃園區'),('桃園市','中壢區'),('桃園市','平鎮區'),('桃園市','蘆竹區'),
    ('台中市','西屯區'),('台中市','北屯區'),('台中市','南屯區'),('台中市','大里區'),
    ('台中市','豐原區'),('台南市','東區'),('台南市','北區'),('台南市','安平區'),
    ('高雄市','三民區'),('高雄市','苓雅區'),('高雄市','前鎮區'),('高雄市','左營區'),
    ('高雄市','鳳山區'),('新竹市','東區'),('新竹市','北區'),('基隆市','仁愛區'),
    ('嘉義市','東區'),('嘉義市','西區'),('宜蘭縣','宜蘭市'),('花蓮縣','花蓮市'),
    ('台東縣','台東市'),('苗栗縣','苗栗市'),('南投縣','南投市'),('彰化縣','彰化市'),
    ('雲林縣','斗六市'),('屏東縣','屏東市'),('澎湖縣','馬公市'),('金門縣','金城鎮'),
    ('新北市','汐止區'),('台北市','萬華區'),
]

store_rows = []
for i, (city, district) in enumerate(cities_districts, start=1):
    days_ago = random.randint(365, 365*4)
    opened = (date(2021,1,1) + timedelta(days=random.randint(0,730))).isoformat()
    store_rows.append((i, f'{city}{district}店', city, district,
                       random.choice([80,120,150,200,250,300]), opened))

cursor.executemany('INSERT INTO stores VALUES (?,?,?,?,?,?)', store_rows)

# 商品資料
categories = {
    '生鮮食品': [('有機高麗菜',12,25),('牛奶 1L',28,45),('雞蛋 10入',35,55),
                ('豬里肌肉 300g',65,98),('鮭魚切片',120,188),('草莓 250g',75,120),
                ('香蕉 1串',20,35),('豆腐',15,28),('雞胸肉 500g',80,125),('白米 2kg',85,138)],
    '烘焙麵包': [('吐司麵包',28,45),('可頌麵包',22,38),('紅豆麵包',18,32),
                ('起司貝果',25,42),('蛋糕切片',55,88)],
    '飲料飲品': [('礦泉水 600ml',5,15),('綠茶 500ml',12,25),('果汁 330ml',18,32),
                ('黑咖啡罐裝',22,38),('運動飲料 600ml',15,28),('養樂多 5入',28,48)],
    '零食點心': [('洋芋片',28,48),('巧克力棒',22,38),('餅乾禮盒',85,138),
                ('果凍 6入',25,42),('糖果包',15,28)],
    '日用清潔': [('洗碗精 500ml',45,72),('沐浴乳 500ml',88,138),('洗髮精 400ml',95,152),
                ('牙膏 120g',28,48),('衛生紙 10捲',65,98),('洗衣精 2kg',155,235)],
    '個人護理': [('護手霜',55,88),('防曬乳 SPF50',188,288),('棉花棒 200支',25,42),
                ('口罩 50入',95,148),('維他命 C 錠',185,298)],
    '冷凍食品': [('水餃 30入',85,135),('湯圓',45,72),('冷凍蝦仁',125,195),
                ('披薩',88,142),('毛豆仁',35,58)],
    '熟食便當': [('雞腿便當',65,98),('排骨便當',60,92),('素食便當',55,85),
                ('炒飯便當',58,88),('關東煮',18,32)],
    '文具用品': [('原子筆',8,18),('筆記本',35,58),('膠帶',15,28),
                ('剪刀',45,72),('資料夾',28,48)],
    '家用雜貨': [('保鮮膜',28,45),('鋁箔紙',22,38),('垃圾袋 50入',35,58),
                ('衣架 10支',38,62),('電池 AA 4入',42,68)],
}

product_rows = []
pid = 1
for cat, items in categories.items():
    for name, cost, sell in items:
        product_rows.append((pid, name, cat, cost, sell))
        pid += 1

cursor.executemany('INSERT INTO products VALUES (?,?,?,?,?)', product_rows)
num_products = len(product_rows)

# 交易資料（約 6000 筆，涵蓋 3 年）
payment_methods = ['現金','信用卡','行動支付','電子票證']
start_date = date(2022, 1, 1)
end_date   = date(2024, 12, 31)
delta_days = (end_date - start_date).days

# 模擬季節性與門店規模差異
def seasonal_weight(d: date) -> float:
    month = d.month
    # 農曆年前後、夏季、年底為旺季
    weights = {1:1.4,2:1.6,3:1.0,4:0.9,5:1.0,6:1.1,
               7:1.2,8:1.2,9:0.9,10:0.9,11:1.1,12:1.5}
    return weights[month]

txn_rows = []
for _ in range(6000):
    store_id   = random.randint(1, 50)
    product_id = random.randint(1, num_products)
    qty        = random.choices([1,2,3,4,5,6], weights=[40,25,15,10,6,4])[0]
    txn_date   = start_date + timedelta(days=random.randint(0, delta_days))
    sw         = seasonal_weight(txn_date)
    # 依季節稍微調整是否新增這筆
    if random.random() > 0.15 * sw:
        price  = product_rows[product_id - 1][4]  # selling_price
        amount = round(price * qty * random.uniform(0.95, 1.05), 0)
        txn_rows.append((
            store_id, product_id, qty, amount,
            txn_date.isoformat(),
            random.choice(payment_methods)
        ))

cursor.executemany(
    'INSERT INTO transactions (store_id,product_id,quantity,amount,txn_date,payment_method) VALUES (?,?,?,?,?,?)',
    txn_rows
)

# 員工資料（每家店 3-8 名員工）
roles = ['店長','副店長','收銀員','理貨員','生鮮專員']
surnames = ['王','李','張','陳','林','吳','劉','黃','趙','周']
given_names = ['小明','小華','志偉','美玲','雅婷','建宏','淑芬','冠廷','思穎','俊傑']
emp_rows = []
eid = 1
for store_id in range(1, 51):
    n_emp = random.randint(3, 8)
    for j in range(n_emp):
        role = roles[j] if j < len(roles) else '收銀員'
        salary = {'店長':65000,'副店長':52000,'收銀員':32000,
                  '理貨員':30000,'生鮮專員':35000}.get(role, 30000)
        salary += random.randint(-2000, 2000)
        hire = (date(2020,1,1) + timedelta(days=random.randint(0,1460))).isoformat()
        name = random.choice(surnames) + random.choice(given_names)
        emp_rows.append((eid, store_id, name, role, hire, salary))
        eid += 1

cursor.executemany('INSERT INTO employees VALUES (?,?,?,?,?,?)', emp_rows)
conn.commit()

# 統計
for table in ['stores','products','transactions','employees']:
    cnt = cursor.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table:<14}: {cnt:>5} 筆')

---
## Part 1：Schema 理解與上下文注入

### 核心概念

LLM 不知道你的資料庫長什麼樣子。**Schema 注入**是 Text-to-SQL 的關鍵步驟：

```
沒有 Schema → LLM 猜測欄位名稱 → SQL 錯誤率極高
有 Schema    → LLM 知道精確欄位 → SQL 準確率顯著提升
```

### SchemaExtractor 的工作
1. 讀取 `sqlite_master` 取得建表 DDL
2. 查詢每張表的行數與樣本資料
3. 組合成結構化的提示文字

In [ ]:
class SchemaExtractor:
    """自動從 SQLite 連線抽取 Schema，並建立 LLM 提示用的上下文字串。"""

    def __init__(self, connection: sqlite3.Connection):
        self.conn = connection

    def get_tables(self) -> list[str]:
        rows = self.conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
        ).fetchall()
        return [r[0] for r in rows]

    def get_table_info(self, table: str) -> dict:
        # 欄位資訊
        cols = self.conn.execute(f'PRAGMA table_info({table})').fetchall()
        columns = [{'name': c[1], 'type': c[2], 'not_null': bool(c[3]),
                    'pk': bool(c[5])} for c in cols]
        # 行數
        count = self.conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        # 樣本（前 3 筆）
        sample_rows = self.conn.execute(
            f'SELECT * FROM {table} LIMIT 3'
        ).fetchall()
        col_names = [c['name'] for c in columns]
        samples = [dict(zip(col_names, r)) for r in sample_rows]
        return {'table': table, 'columns': columns,
                'row_count': count, 'samples': samples}

    def build_schema_context(self) -> str:
        lines = ['## 資料庫 Schema\n']
        for table in self.get_tables():
            info = self.get_table_info(table)
            lines.append(f'### 表格：{table}  （共 {info["row_count"]:,} 筆）')
            lines.append('| 欄位名稱 | 類型 | PK | 非空 |')
            lines.append('|---------|------|----|----|')
            for c in info['columns']:
                pk  = '✓' if c['pk']       else ''
                nn  = '✓' if c['not_null'] else ''
                lines.append(f"| {c['name']} | {c['type']} | {pk} | {nn} |")
            lines.append('')
            # 樣本資料
            if info['samples']:
                sample_str = json.dumps(info['samples'][0], ensure_ascii=False)
                lines.append(f'樣本資料（第 1 筆）：`{sample_str}`')
            lines.append('')
        return '\n'.join(lines)


schema_extractor = SchemaExtractor(conn)
schema_context   = schema_extractor.build_schema_context()
print(schema_context)

In [ ]:
# ── 對照實驗：有無 Schema 注入的差異 ──────────────────────────────────────
question = '哪個城市的總銷售額最高？'

def ask_sql(question: str, include_schema: bool) -> str:
    system = '你是一個 SQL 專家。請只輸出可執行的 SQLite SQL 查詢，不含任何解釋。'
    if include_schema:
        system += f'\n\n{schema_context}'
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': question}
        ],
        temperature=0
    )
    return resp.choices[0].message.content.strip()

sql_no_schema   = ask_sql(question, include_schema=False)
sql_with_schema = ask_sql(question, include_schema=True)

print('問題：', question)
print('\n【沒有 Schema 注入】')
print(sql_no_schema)
print('\n【有 Schema 注入】')
print(sql_with_schema)

# 嘗試執行有 Schema 的版本
try:
    df_test = pd.read_sql_query(sql_with_schema, conn)
    print('\n執行結果：')
    print(df_test.head())
except Exception as e:
    print(f'執行失敗：{e}')

---
## Part 2：Text-to-SQL 核心實作

### 架構說明

```
TextToSQLAgent
    ├── SQLValidator   — 攔截 DROP/DELETE/UPDATE/INSERT
    ├── SQLExecutor    — 執行並回傳 DataFrame
    └── ErrorRecovery  — 最多 2 次自動修正
```

### SQL 安全驗證

在任何 AI 生成的 SQL 執行前，必須先通過白名單驗證：
- **允許**：`SELECT`、`WITH`（CTE）
- **拒絕**：`DROP`、`DELETE`、`UPDATE`、`INSERT`、`ALTER`、`TRUNCATE`

In [ ]:
@dataclass
class SQLValidationResult:
    is_safe: bool
    cleaned_sql: str
    error_msg: str = ''


class SQLValidator:
    """驗證 LLM 產生的 SQL 是否安全可執行。"""

    DANGEROUS_PATTERNS = [
        r'\bDROP\b', r'\bDELETE\b', r'\bUPDATE\b',
        r'\bINSERT\b', r'\bALTER\b', r'\bTRUNCATE\b',
        r'\bCREATE\b', r'\bREPLACE\b', r'\bATTACH\b',
        r'--',          # SQL 行注釋（注入跡象）
        r';\s*SELECT',  # 多語句注入
    ]

    MAX_RESULT_ROWS = 500   # 防止全表掃描回傳過多資料

    def validate(self, sql: str) -> SQLValidationResult:
        # 1. 清除 Markdown code fence
        sql = re.sub(r'^```[\w]*\n?', '', sql.strip())
        sql = re.sub(r'```$', '', sql.strip()).strip()

        upper = sql.upper()

        # 2. 危險關鍵字檢查
        for pattern in self.DANGEROUS_PATTERNS:
            if re.search(pattern, upper):
                return SQLValidationResult(
                    is_safe=False, cleaned_sql='',
                    error_msg=f'危險操作已攔截：包含 {pattern}'
                )

        # 3. 必須以 SELECT 或 WITH 開頭
        if not re.match(r'^\s*(SELECT|WITH)', upper):
            return SQLValidationResult(
                is_safe=False, cleaned_sql='',
                error_msg='只允許 SELECT 查詢'
            )

        # 4. 自動加入 LIMIT 保護（若無）
        if 'LIMIT' not in upper:
            sql = sql.rstrip(';') + f' LIMIT {self.MAX_RESULT_ROWS}'

        return SQLValidationResult(is_safe=True, cleaned_sql=sql)


validator = SQLValidator()

# 測試
test_cases = [
    "SELECT * FROM stores",
    "DROP TABLE transactions",
    "DELETE FROM employees WHERE salary < 30000",
    "SELECT store_id, SUM(amount) FROM transactions GROUP BY store_id",
    "SELECT * FROM stores; DROP TABLE stores",
]
print(f'{'SQL':<55} {'安全？':<8} 說明')
print('-' * 90)
for sql in test_cases:
    r = validator.validate(sql)
    status = '✓ 安全' if r.is_safe else '✗ 攔截'
    note   = r.error_msg if not r.is_safe else '通過'
    print(f'{sql[:53]:<55} {status:<8} {note}')

In [ ]:
@dataclass
class QueryResult:
    question:    str
    sql:         str
    explanation: str
    dataframe:   Optional[pd.DataFrame]
    success:     bool
    error:       str = ''
    retries:     int = 0


class TextToSQLAgent:
    """自然語言 → SQL → DataFrame，含自動錯誤修正。"""

    MAX_RETRIES = 2

    def __init__(self, connection: sqlite3.Connection, schema_ctx: str):
        self.conn      = connection
        self.schema    = schema_ctx
        self.validator = SQLValidator()

    def _generate_sql(self, question: str,
                      error_context: str = '') -> Tuple[str, str]:
        system_prompt = f"""你是一個專業的 SQLite SQL 生成助理。
請根據以下資料庫 Schema 將使用者問題轉換為 SQL 查詢。

{self.schema}

規則：
1. 只能使用 SELECT 查詢
2. 使用 SQLite 語法（日期函數用 DATE(), STRFTIME()）
3. 欄位名稱必須完全符合 Schema
4. 請以 JSON 格式回覆：{{"sql": "...", "explanation": "..."}}
5. explanation 請用繁體中文說明這個查詢做了什麼
"""
        user_content = question
        if error_context:
            user_content += f'\n\n[前次查詢失敗：{error_context}，請修正 SQL]'

        resp = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_content}
            ],
            temperature=0,
            response_format={'type': 'json_object'}
        )
        content = json.loads(resp.choices[0].message.content)
        return content.get('sql', ''), content.get('explanation', '')

    def query(self, question: str) -> QueryResult:
        error_ctx = ''
        for attempt in range(self.MAX_RETRIES + 1):
            sql, explanation = self._generate_sql(question, error_ctx)

            # 安全驗證
            vr = self.validator.validate(sql)
            if not vr.is_safe:
                return QueryResult(question=question, sql=sql,
                                   explanation='', dataframe=None,
                                   success=False, error=vr.error_msg)

            # 執行
            try:
                df = pd.read_sql_query(vr.cleaned_sql, self.conn)
                return QueryResult(question=question, sql=vr.cleaned_sql,
                                   explanation=explanation, dataframe=df,
                                   success=True, retries=attempt)
            except Exception as e:
                error_ctx = str(e)
                if attempt == self.MAX_RETRIES:
                    return QueryResult(question=question, sql=sql,
                                       explanation='', dataframe=None,
                                       success=False, error=error_ctx,
                                       retries=attempt)

        # 不應到達此處
        return QueryResult(question=question, sql='', explanation='',
                           dataframe=None, success=False)


agent = TextToSQLAgent(conn, schema_context)
print('✓ TextToSQLAgent 初始化完成')

In [ ]:
# ── 8 個真實業務問題示範 ──────────────────────────────────────────────────
business_questions = [
    '哪五家門店的總銷售額最高？',
    '2023 年各季度的總銷售額趨勢如何？',
    '哪個商品品類的利潤率最高？',
    '行動支付佔所有交易的比例是多少？',
    '平均客單價最高的三個城市是哪些？',
    '2024 年每個月銷售額最高的商品是什麼？',
    '員工人數超過 6 人的門店清單',
    '台北市各門店的平均交易金額',
]

results = {}
for q in business_questions:
    print(f'\n問：{q}')
    r = agent.query(q)
    results[q] = r
    if r.success:
        print(f'SQL：{r.sql[:120]}...' if len(r.sql) > 120 else f'SQL：{r.sql}')
        print(f'說明：{r.explanation}')
        print(r.dataframe.head(5).to_string(index=False))
        if r.retries:
            print(f'（自動修正 {r.retries} 次後成功）')
    else:
        print(f'失敗：{r.error}')

---
## Part 3：結果解讀（Result Interpretation）

查到數字只是第一步。業務人員需要的是**洞察**，而非原始數字。

`ResultInterpreter` 將 (問題 + SQL + DataFrame) 轉換為：
- 關鍵指標的白話說明
- 比較性描述（「高於/低於平均 X%」）
- 可能的業務含義
- 資料限制說明（若有）

In [ ]:
class ResultInterpreter:
    """將 SQL 查詢結果轉換為自然語言業務洞察。"""

    def __init__(self, client: OpenAI, model: str = CHAT_MODEL):
        self.client = client
        self.model  = model

    def _detect_result_type(self, df: pd.DataFrame) -> str:
        if df.shape == (1, 1):
            return 'single_value'
        if df.shape[0] <= 3 and df.shape[1] <= 3:
            return 'small_table'
        # 判斷是否有日期欄位（時間序列）
        for col in df.columns:
            if any(k in col.lower() for k in ['date','month','year','quarter','季','月','年']):
                return 'time_series'
        return 'comparison_table'

    def interpret(self, question: str, sql: str,
                  df: pd.DataFrame) -> str:
        result_type = self._detect_result_type(df)

        # 把 DataFrame 轉成文字（最多 20 列）
        df_text = df.head(20).to_string(index=False)

        prompt = f"""你是一名資深零售業務分析師。
請根據以下查詢結果提供業務洞察，使用繁體中文，不超過 200 字。

原始問題：{question}
結果類型：{result_type}
查詢結果：
{df_text}

請提供：
1. 核心發現（1-2 句）
2. 業務含義（1-2 句）
3. 建議行動（1 句，若適用）

若資料樣本可能不完整，請說明限制。"""

        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.3
        )
        return resp.choices[0].message.content.strip()


interpreter = ResultInterpreter(client)

# 示範 3 個問題的完整解讀
demo_qs = list(results.keys())[:3]
for q in demo_qs:
    r = results[q]
    if r.success and r.dataframe is not None and len(r.dataframe) > 0:
        print(f'\n{'='*60}')
        print(f'問題：{q}')
        print(f'\n查詢結果摘要：')
        print(r.dataframe.head().to_string(index=False))
        insight = interpreter.interpret(q, r.sql, r.dataframe)
        print(f'\n業務洞察：')
        print(insight)

---
## Part 4：多步驟分析推理（Multi-step Analysis）

### 為什麼需要多步驟？

某些業務問題無法用單一 SQL 回答，例如：

> 「績效墊底的 5 家門店，主要是哪個品類拖累了銷售？和頭部 5 家比較呢？」

這需要：
1. 先找出銷售額最低的 5 家門店（Query 1）
2. 查這 5 家的品類銷售分布（Query 2，依賴 Q1 結果）
3. 找出銷售額最高的 5 家（Query 3）
4. 比較兩組的品類差異（Query 4）

`AnalyticalPlannerAgent` 自動完成這個拆解流程。

In [ ]:
@dataclass
class AnalysisStep:
    step_no:     int
    description: str
    sql:         str
    result:      Optional[pd.DataFrame] = None
    success:     bool = False
    error:       str  = ''


class AnalyticalPlannerAgent:
    """將複雜業務問題拆解為有序子查詢，逐步執行並彙整結論。"""

    def __init__(self, connection: sqlite3.Connection,
                 schema_ctx: str, client: OpenAI):
        self.conn      = connection
        self.schema    = schema_ctx
        self.client    = client
        self.validator = SQLValidator()

    def plan(self, complex_question: str) -> list[AnalysisStep]:
        """請 LLM 將問題拆解為子步驟清單（含 SQL）。"""
        prompt = f"""你是資深資料分析師，請將以下複雜分析問題拆解為 2-5 個有序查詢步驟。

資料庫 Schema：
{self.schema}

複雜問題：{complex_question}

請以 JSON 陣列回覆，每個步驟包含：
- step_no: 步驟編號（從 1 開始）
- description: 繁體中文說明此步驟目的
- sql: SQLite SELECT 查詢（不可有 DROP/DELETE/UPDATE）

回覆格式：{{"steps": [...]}}"""

        resp = self.client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
            response_format={'type': 'json_object'}
        )
        data  = json.loads(resp.choices[0].message.content)
        steps = []
        for s in data.get('steps', []):
            steps.append(AnalysisStep(
                step_no     = s['step_no'],
                description = s['description'],
                sql         = s['sql']
            ))
        return steps

    def execute_plan(self, steps: list[AnalysisStep]) -> list[AnalysisStep]:
        """依序執行每個步驟。"""
        for step in steps:
            vr = self.validator.validate(step.sql)
            if not vr.is_safe:
                step.success = False
                step.error   = vr.error_msg
                continue
            try:
                step.result  = pd.read_sql_query(vr.cleaned_sql, self.conn)
                step.success = True
            except Exception as e:
                step.success = False
                step.error   = str(e)
        return steps

    def synthesize(self, question: str,
                   steps: list[AnalysisStep]) -> str:
        """根據所有步驟結果產生最終結論。"""
        context_parts = [f'原始問題：{question}\n']
        for s in steps:
            context_parts.append(f'步驟 {s.step_no}：{s.description}')
            if s.success and s.result is not None:
                context_parts.append(s.result.head(8).to_string(index=False))
            else:
                context_parts.append(f'（執行失敗：{s.error}）')
            context_parts.append('')

        synthesis_prompt = '\n'.join(context_parts) + \
            '\n請用繁體中文提供一份 150 字以內的綜合分析結論，包含關鍵發現與建議。'

        resp = self.client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[{'role': 'user', 'content': synthesis_prompt}],
            temperature=0.3
        )
        return resp.choices[0].message.content.strip()


planner = AnalyticalPlannerAgent(conn, schema_context, client)

# ── 示範：複雜分析問題 ───────────────────────────────────────────────────
complex_q = '績效墊底的 5 家門店，主要是哪個商品品類拖累銷售？和頭部 5 家相比有何差異？'

print(f'複雜問題：{complex_q}\n')
print('─' * 50)
print('步驟規劃中...')
steps = planner.plan(complex_q)

for s in steps:
    print(f'\n步驟 {s.step_no}：{s.description}')
    print(f'SQL：{s.sql[:100]}...' if len(s.sql) > 100 else f'SQL：{s.sql}')

print('\n' + '─' * 50)
print('執行各步驟...')
steps = planner.execute_plan(steps)

for s in steps:
    print(f'\n步驟 {s.step_no}（{"成功" if s.success else "失敗"}）：')
    if s.success and s.result is not None:
        print(s.result.head(5).to_string(index=False))

print('\n' + '─' * 50)
print('綜合結論：')
conclusion = planner.synthesize(complex_q, steps)
print(conclusion)

---
## Part 5：自然語言報表生成

`ReportGeneratorAgent` 針對一個業務問題，自動生成包含以下章節的迷你報表：

1. 執行摘要（Executive Summary）
2. 關鍵指標
3. 趨勢分析
4. 表現最佳 / 最差的門店或商品
5. 行動建議

In [ ]:
class ReportGeneratorAgent:
    """根據業務問題自動生成多節段分析報表。"""

    REPORT_QUERIES = [
        ('本月各門店銷售總額排名（前10）',
         """SELECT s.store_name, s.city,
                   ROUND(SUM(t.amount),0) AS total_sales,
                   COUNT(*) AS txn_count
            FROM transactions t
            JOIN stores s ON t.store_id = s.store_id
            WHERE STRFTIME('%Y-%m', t.txn_date) =
                  STRFTIME('%Y-%m', DATE('now','-1 month'))
            GROUP BY s.store_id
            ORDER BY total_sales DESC LIMIT 10"""),
        ('近 3 個月各品類銷售趨勢',
         """SELECT STRFTIME('%Y-%m', t.txn_date) AS month,
                   p.category,
                   ROUND(SUM(t.amount),0) AS sales
            FROM transactions t
            JOIN products p ON t.product_id = p.product_id
            WHERE t.txn_date >= DATE('now','-3 months')
            GROUP BY month, p.category
            ORDER BY month, sales DESC LIMIT 50"""),
        ('整體 KPI 摘要',
         """SELECT
              ROUND(SUM(amount),0)      AS 總銷售額,
              COUNT(*)                   AS 總交易筆數,
              ROUND(AVG(amount),1)       AS 平均客單價,
              COUNT(DISTINCT store_id)   AS 活躍門店數,
              COUNT(DISTINCT product_id) AS 活躍商品數
            FROM transactions"""),
    ]

    def __init__(self, connection: sqlite3.Connection, client: OpenAI):
        self.conn      = connection
        self.client    = client
        self.validator = SQLValidator()

    def _run_query(self, sql: str) -> Optional[pd.DataFrame]:
        vr = self.validator.validate(sql)
        if not vr.is_safe:
            return None
        try:
            return pd.read_sql_query(vr.cleaned_sql, self.conn)
        except Exception:
            return None

    def generate(self, report_title: str) -> str:
        # 執行所有預設查詢
        data_sections = []
        for label, sql in self.REPORT_QUERIES:
            df = self._run_query(sql)
            if df is not None and len(df) > 0:
                data_sections.append(f'--- {label} ---')
                data_sections.append(df.head(10).to_string(index=False))

        data_context = '\n'.join(data_sections)

        prompt = f"""你是資深零售業分析師，請根據以下資料生成一份繁體中文業務報表。

報表標題：{report_title}

資料來源：
{data_context}

請生成包含以下章節的報表（總長度 300-400 字）：
## 執行摘要
## 關鍵指標
## 趨勢分析
## 表現亮點與落後
## 行動建議"""

        resp = self.client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.4
        )
        return resp.choices[0].message.content.strip()


report_agent = ReportGeneratorAgent(conn, client)
report = report_agent.generate('2024 年度門店績效月報')
print(report)

In [ ]:
# ── 配套視覺化：各品類月銷售趨勢 ───────────────────────────────────────────
trend_sql = """
SELECT STRFTIME('%Y-%m', t.txn_date) AS month,
       p.category,
       ROUND(SUM(t.amount)/1000.0, 1) AS sales_k
FROM transactions t
JOIN products p ON t.product_id = p.product_id
WHERE t.txn_date >= DATE('2024-01-01')
GROUP BY month, p.category
ORDER BY month
"""
df_trend = pd.read_sql_query(trend_sql, conn)

if len(df_trend) > 0:
    pivot = df_trend.pivot_table(
        index='month', columns='category',
        values='sales_k', aggfunc='sum'
    ).fillna(0)

    fig, ax = plt.subplots(figsize=(14, 6))
    pivot.plot(ax=ax, marker='o', linewidth=1.5)
    ax.set_title('2024 年各商品品類月銷售趨勢（千元）', fontsize=14)
    ax.set_xlabel('月份')
    ax.set_ylabel('銷售額（千元）')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('（2024 年資料不足，請確認模擬資料日期範圍）')

---
## Part 6：安全邊界與權限控制

### 三層防護架構

```
層級 1：QueryScopeFilter  — 角色可見表格 / 欄位白名單
層級 2：RowLevelSecurity  — 門店店長只能看自己門店
層級 3：ComplexityScorer  — 警告潛在的慢查詢
```

In [ ]:
# ── 角色定義 ──────────────────────────────────────────────────────────────
ROLE_PERMISSIONS = {
    'analyst': {
        'tables': ['stores','products','transactions','employees'],
        'forbidden_columns': ['salary'],  # 分析師看不到薪資
    },
    'store_manager': {
        'tables': ['stores','products','transactions'],
        'forbidden_columns': ['salary','emp_id'],
    },
    'marketing': {
        'tables': ['stores','products','transactions'],
        'forbidden_columns': ['salary','cost_price'],  # 行銷不看成本
    },
}


class QueryScopeFilter:
    """根據使用者角色過濾 SQL 中存取的資料範圍。"""

    def __init__(self, role: str, store_id: Optional[int] = None):
        self.role     = role
        self.store_id = store_id  # None = 所有門店
        self.perms    = ROLE_PERMISSIONS.get(role, {'tables': [], 'forbidden_columns': []})

    def check_sql(self, sql: str) -> Tuple[bool, str]:
        upper = sql.upper()

        # 1. 資料表存取檢查
        for table in ['stores','products','transactions','employees']:
            if table.upper() in upper and table not in self.perms['tables']:
                return False, f'角色 [{self.role}] 無權存取資料表：{table}'

        # 2. 欄位存取檢查
        for col in self.perms.get('forbidden_columns', []):
            if col.upper() in upper:
                return False, f'角色 [{self.role}] 無權存取欄位：{col}'

        return True, ''

    def inject_row_security(self, sql: str) -> str:
        """若為門店店長且 store_id 已設定，自動加入 WHERE 子句。"""
        if self.role == 'store_manager' and self.store_id is not None:
            # 簡易注入（生產環境應使用參數化查詢）
            sql = sql.rstrip(';')
            if 'WHERE' in sql.upper():
                sql += f' AND transactions.store_id = {self.store_id}'
            elif 'FROM transactions' in sql.upper() or 'JOIN transactions' in sql.upper():
                sql += f' WHERE transactions.store_id = {self.store_id}'
        return sql


class QueryComplexityScorer:
    """評估查詢複雜度，對潛在慢查詢發出警告。"""

    def score(self, sql: str) -> dict:
        upper   = sql.upper()
        score   = 0
        reasons = []

        join_count = upper.count('JOIN')
        score += join_count * 20
        if join_count > 2:
            reasons.append(f'{join_count} 個 JOIN，可能較慢')

        if 'LIKE' in upper and upper.count('%') >= 2:
            score += 30
            reasons.append('雙向 LIKE 模糊搜尋，無法使用索引')

        if 'SELECT *' in upper:
            score += 15
            reasons.append('SELECT * 回傳所有欄位')

        if 'SUBQUERY' in upper or upper.count('SELECT') > 2:
            score += 25
            reasons.append('多層子查詢')

        level = 'LOW' if score < 30 else ('MEDIUM' if score < 60 else 'HIGH')
        return {'score': score, 'level': level, 'reasons': reasons}


# ── 測試 ──────────────────────────────────────────────────────────────────
test_sqls = [
    ("SELECT store_name, salary FROM employees JOIN stores ON employees.store_id=stores.store_id",
     'analyst'),
    ("SELECT * FROM transactions WHERE store_id=5",
     'marketing'),
    ("SELECT amount, txn_date FROM transactions WHERE store_id=3",
     'store_manager'),
]

scorer = QueryComplexityScorer()
print(f'{'SQL':<60} {'角色':<15} {'通過？':<8} {'複雜度'}')
print('-' * 110)
for sql, role in test_sqls:
    scope_filter = QueryScopeFilter(role, store_id=3 if role=='store_manager' else None)
    ok, msg      = scope_filter.check_sql(sql)
    complexity   = scorer.score(sql)
    status = '✓' if ok else f'✗ {msg}'
    print(f'{sql[:58]:<60} {role:<15} {status[:7]:<8} {complexity["level"]}({complexity["score"]})')

---
## Part 7：生產部署考量

### 7.1 查詢快取（Semantic Cache）

同語意的問題不必每次呼叫 LLM，可直接回傳快取結果。

In [ ]:
import hashlib
from datetime import datetime


@dataclass
class CacheEntry:
    question:   str
    sql:        str
    result_csv: str
    embedding:  list[float]
    created_at: datetime = field(default_factory=datetime.now)
    ttl_minutes: int = 60

    def is_expired(self) -> bool:
        age = (datetime.now() - self.created_at).total_seconds() / 60
        return age > self.ttl_minutes


class SemanticQueryCache:
    """基於語意相似度的查詢快取，避免重複呼叫 LLM。"""

    def __init__(self, similarity_threshold: float = 0.92):
        self.entries   = []
        self.threshold = similarity_threshold

    def _embed(self, text: str) -> list[float]:
        resp = client.embeddings.create(model=EMBED_MODEL, input=text)
        return resp.data[0].embedding

    @staticmethod
    def _cosine_sim(a: list[float], b: list[float]) -> float:
        import math
        dot   = sum(x*y for x,y in zip(a,b))
        norm_a = math.sqrt(sum(x**2 for x in a))
        norm_b = math.sqrt(sum(x**2 for x in b))
        return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

    def lookup(self, question: str) -> Optional[CacheEntry]:
        if not self.entries:
            return None
        q_emb = self._embed(question)
        best_score = 0.0
        best_entry = None
        for entry in self.entries:
            if entry.is_expired():
                continue
            score = self._cosine_sim(q_emb, entry.embedding)
            if score > best_score:
                best_score = score
                best_entry = entry
        if best_score >= self.threshold:
            print(f'  [快取命中] 相似度={best_score:.3f}')
            return best_entry
        return None

    def store(self, question: str, sql: str,
              df: pd.DataFrame, ttl_minutes: int = 60):
        emb   = self._embed(question)
        entry = CacheEntry(
            question=question, sql=sql,
            result_csv=df.to_csv(index=False),
            embedding=emb, ttl_minutes=ttl_minutes
        )
        self.entries.append(entry)


cache = SemanticQueryCache()

# 存入一個查詢
q1   = '哪五家門店銷售最好？'
r1   = results.get('哪五家門店的總銷售額最高？')
if r1 and r1.success:
    cache.store(q1, r1.sql, r1.dataframe)
    print(f'快取已儲存：{q1}')

# 用語意相似問題查找
q2 = '銷售業績排名前五的門店是哪些？'
print(f'\n查詢：{q2}')
hit = cache.lookup(q2)
if hit:
    df_cached = pd.read_csv(__import__('io').StringIO(hit.result_csv))
    print(f'快取命中！原始問題：{hit.question}')
    print(df_cached.head())
else:
    print('快取未命中，需呼叫 LLM')

In [ ]:
# ── 7.2 大結果集分頁 ─────────────────────────────────────────────────────
def paginate_query(sql: str, page: int, page_size: int = 20) -> pd.DataFrame:
    """對 SQL 查詢結果進行分頁，避免單次回傳過多資料。"""
    offset    = (page - 1) * page_size
    paged_sql = sql.rstrip(';') + f' LIMIT {page_size} OFFSET {offset}'
    return pd.read_sql_query(paged_sql, conn)

base_sql = 'SELECT txn_id, store_id, amount, txn_date FROM transactions ORDER BY txn_date DESC'

page1 = paginate_query(base_sql, page=1)
page2 = paginate_query(base_sql, page=2)
print(f'第 1 頁（共 {len(page1)} 筆）：')
print(page1.head(3).to_string(index=False))
print(f'\n第 2 頁（共 {len(page2)} 筆）：')
print(page2.head(3).to_string(index=False))

# ── 7.3 連接真實資料庫（概念說明）────────────────────────────────────────
print("""
─── 連接真實資料庫（概念）───────────────────────────────────

PostgreSQL：
    import psycopg2
    conn = psycopg2.connect(host=..., dbname=..., user=..., password=...)
    df = pd.read_sql_query(sql, conn)

Google BigQuery：
    from google.cloud import bigquery
    bq = bigquery.Client(project='my-project')
    df = bq.query(sql).to_dataframe()

Snowflake：
    import snowflake.connector
    conn = snowflake.connector.connect(account=..., user=..., password=...)
    df = pd.read_sql_query(sql, conn)

關鍵原則：
    - 永遠使用唯讀角色（只有 SELECT 權限）
    - 參數化查詢防止 SQL 注入
    - 設定查詢超時（timeout）
    - 啟用查詢稽核日誌
""")

---
## Part 8：面試 Q&A

以下 5 題是 Text-to-SQL 與 BI Agent 面試中最常見的深度問題。

In [ ]:
qa_pairs = [
    {
        'q': 'Q1：Text-to-SQL 系統在實際應用中最常遇到的準確性挑戰是什麼？如何緩解？',
        'a': """
A1：主要挑戰有三類：

① 模糊欄位名稱
   問題：「過去一個月」→ LLM 不知道日期欄位叫 txn_date 還是 created_at
   解法：Schema 注入時加入欄位描述與樣本值（如 "txn_date: '2024-03-15'"）

② 跨表 JOIN 關係推斷
   問題：「台北市的銷售額」需要知道 transactions → stores 透過 store_id 關聯
   解法：在 Schema 上下文中明確標示外鍵關係與 JOIN 路徑

③ 業務術語對應
   問題：「客單價」= amount/COUNT(*)；「毛利率」= (selling_price-cost_price)/selling_price
   解法：建立業務詞彙表（Business Glossary）注入提示，或用 Few-shot 示範

衡量標準：Execution Accuracy（SQL 能執行且結果正確）通常比 Exact Match 更重要。
"""
    },
    {
        'q': 'Q2：使用者問了「最近業績怎麼樣？」這種模糊問題，AI 系統應該如何處理？',
        'a': """
A2：三段式模糊問題處理流程：

① 意圖澄清（Clarification）
   先詢問：「您想看哪個時間範圍？哪家門店？哪個品類？」
   可設定預設值：「未指定則預設近 30 天、全部門店」

② 假設說明（Assumption Declaration）
   在回答前告知：「以下分析基於：過去 30 天、全台 50 家門店、所有品類」

③ 補充建議（Follow-up Suggestions）
   主動提示可深入的方向：「是否想進一步看各城市比較？或特定品類趨勢？」

這種設計讓 AI 更像顧問，而非只是查詢工具。
"""
    },
    {
        'q': 'Q3：AI 生成 SQL 系統中，如何有效防止 SQL 注入攻擊？',
        'a': """
A3：多層防護缺一不可：

① LLM Prompt 層
   明確指示「只生成 SELECT 查詢」並在 system prompt 設定嚴格規則

② 正規表達式驗證層（本筆記本 SQLValidator）
   執行前用 regex 攔截 DROP/DELETE/UPDATE/INSERT/ALTER 等危險關鍵字
   拒絕包含 -- 注釋或多語句（;SELECT）的查詢

③ 資料庫使用者權限層（最重要）
   系統使用的 DB 帳號只授予 SELECT 權限
   即使 Prompt 被繞過，資料庫層也會拒絕寫入操作

④ 查詢沙盒層
   設定最大執行時間（query timeout）
   限制回傳行數（LIMIT 保護）

最重要的原則：永遠不要信任 AI 生成的輸出，即使模型是可信的。
"""
    },
    {
        'q': 'Q4：多步驟分析推理（Chained Queries）的主要難點是什麼？如何設計穩健的系統？',
        'a': """
A4：核心難點與解法：

難點 1：中間結果傳遞
   前一步的結果（如門店 ID 清單）需動態注入後一步的 SQL
   解法：AnalyticalPlannerAgent 用 Python 將 DataFrame 結果轉為 IN (1,2,3,...) 子句

難點 2：步驟順序依賴
   步驟 3 依賴步驟 2，步驟 2 依賴步驟 1 → 任一失敗影響全局
   解法：實作步驟狀態機，失敗時自動用備選查詢策略

難點 3：Token 使用量爆炸
   每步驟結果都送回 LLM → 大表結果會超出 context window
   解法：中間結果只傳摘要（前 10 行 + 統計資訊），不傳完整 DataFrame

難點 4：使用者等待時間
   解法：串流輸出每個步驟的進度，讓使用者知道系統在思考中
"""
    },
    {
        'q': 'Q5：如何讓 BI 工具真正對非技術人員（店長、行銷）友善？有哪些 UX 設計考量？',
        'a': """
A5：非技術使用者友善設計的五個原則：

① 問題建議（Query Suggestions）
   根據使用者角色預設 5-10 個常見問題模板，降低空白頁焦慮
   例：店長看到「上週我的門店業績」、「本月最暢銷商品 Top 5」

② 結果解讀自動化
   不只顯示數字，自動生成「這個數字比上期高 15%，在全台排名第 3」
   使用視覺化暗示（上升箭頭、顏色）強化資訊傳遞

③ 錯誤訊息人性化
   不顯示 SQL error，改為「找不到相關資料，您是否想查詢...？」

④ 對話式追問
   每個回答後提供 2-3 個相關後續問題，引導使用者深入探索

⑤ 結果可信度標示
   說明「這是基於完整 3 年資料」或「最近 7 天資料可能尚未完全更新」
   讓非技術使用者理解資料的局限性
"""
    },
]

for pair in qa_pairs:
    print(f"\n{'='*70}")
    print(pair['q'])
    print(pair['a'])

---
## 學習重點總結

| 主題 | 核心技術 | 重要程度 |
|------|---------|----------|
| Schema 注入 | 自動抽取 DDL + 樣本資料注入 Prompt | ★★★★★ |
| SQL 驗證 | Regex 白名單 + DB 唯讀帳號 | ★★★★★ |
| 錯誤自修正 | 錯誤訊息回饋 LLM 最多 2 次重試 | ★★★★☆ |
| 結果解讀 | (問題 + SQL + DataFrame) → 自然語言洞察 | ★★★★☆ |
| 多步驟推理 | 問題拆解 → 有序執行 → 綜合結論 | ★★★★☆ |
| 語意快取 | Embedding 相似度 + TTL 過期機制 | ★★★☆☆ |
| 權限控制 | 角色白名單 + Row-level Security | ★★★★★ |

### 延伸閱讀方向
- **Spider / BIRD Benchmark** — 評估 Text-to-SQL 模型的標準資料集
- **DAIL-SQL** — 在 Spider 上達到 SOTA 的 few-shot Text-to-SQL 方法
- **DIN-SQL** — 將複雜 SQL 分解為子問題的方法論
- **Vanna.ai** — 生產級 Text-to-SQL 開源框架（支援 RAG + Fine-tuning）

### 下一步
- **NB18**：多模態 RAG — 圖片、表格、PDF 混合查詢
- **NB19**：RAG 系統的 A/B 測試與持續改善框架